# V10 Foundation-Model Triplet — Chronos zero-shot

Run Amazon's Chronos-T5-Small (2024) on each (Партнер, Артикул) monthly
shipment series and produce zero-shot forecasts for the validation +
test windows.  Chronos was trained on M-series competitions, electricity,
traffic, weather, and finance — never seen toy-distribution data, so its
residuals are guaranteed orthogonal to V1-V9.

Output:
* `/kaggle/working/preds_v10_chronos_val.csv`
* `/kaggle/working/preds_v10_chronos_test.csv`

Pulled into `output/` after the kernel completes.

In [ ]:
%pip uninstall -y -q torchvision  # avoid torchvision::nms import trigger
%pip install --quiet --upgrade transformers
%pip install --quiet chronos-forecasting==1.5.2
import sys, os, glob, time
import numpy as np, pandas as pd, torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
data_root = '/kaggle/input'
candidates = glob.glob(f'{data_root}/*/abt_v9_cached.parquet') + glob.glob(f'{data_root}/*/abt_v10_cached.parquet') + glob.glob(f'{data_root}/*/*.parquet')
print('candidates:', candidates[:5])
abt_path = next((p for p in candidates if p.endswith('abt_v10_cached.parquet') or p.endswith('abt_v9_cached.parquet')), candidates[0])
print('using ABT:', abt_path)
abt = pd.read_parquet(abt_path)
print('ABT shape:', abt.shape)
print(abt[['Период','Партнер','Артикул','target_qty_imputed']].head())

In [ ]:
# Build per-(partner,sku) monthly history; identify val + test rows
abt['Период_p'] = pd.PeriodIndex(abt['Период'].astype(str), freq='M')
val_periods = pd.period_range('2024-07', '2025-06', freq='M')
test_periods = pd.period_range('2025-07', '2026-01', freq='M')
train_periods = pd.period_range('2020-01', '2024-06', freq='M')

active_pairs = abt[abt['Период_p'].isin(val_periods) | abt['Период_p'].isin(test_periods)][['Партнер','Артикул']].drop_duplicates()
print('active pairs:', len(active_pairs))

# Pivot to (pair, month) wide
wide = (abt.groupby(['Партнер','Артикул','Период_p'], observed=True)['target_qty_imputed'].sum()
        .reset_index().pivot_table(index=['Партнер','Артикул'], columns='Период_p', values='target_qty_imputed', fill_value=0))
wide = wide.reindex(pd.MultiIndex.from_frame(active_pairs))
wide = wide.fillna(0).astype(np.float32)
print('wide shape:', wide.shape)

In [ ]:
from chronos import ChronosPipeline
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Loading Chronos-T5-Small ...')
pipe = ChronosPipeline.from_pretrained('amazon/chronos-t5-small', device_map=device, torch_dtype=torch.float16 if device == 'cuda' else torch.float32)
print('OK')

In [ ]:
from tqdm.auto import tqdm
context_len = 36  # 3 years of monthly history
all_periods = list(train_periods) + list(val_periods) + list(test_periods)

predictions = {p: np.zeros(len(wide), dtype=np.float32) for p in list(val_periods) + list(test_periods)}

rows = wide.values
col_to_ix = {c: i for i, c in enumerate(wide.columns)}
n_pairs = len(rows)

BATCH = 256
all_target_periods = list(val_periods) + list(test_periods)

def context_for(pair_ix, target_period):
    end_ix = col_to_ix[target_period] - 1
    start_ix = max(0, end_ix - context_len + 1)
    return rows[pair_ix, start_ix:end_ix + 1]

for tgt in tqdm(all_target_periods, desc='periods'):
    contexts = [context_for(i, tgt) for i in range(n_pairs)]
    for batch_start in range(0, n_pairs, BATCH):
        batch = [torch.tensor(c, dtype=torch.float32) for c in contexts[batch_start:batch_start+BATCH]]
        with torch.no_grad():
            forecast = pipe.predict(context=batch, prediction_length=1, num_samples=20)
        median = np.median(forecast.cpu().numpy().astype(np.float32), axis=1).reshape(-1)
        median = np.clip(median, 0, None)
        predictions[tgt][batch_start:batch_start+len(batch)] = median
print('done forecasting')

In [ ]:
out_rows = []
for tgt, preds in predictions.items():
    for i, (partner, sku) in enumerate(wide.index):
        out_rows.append({'Период': str(tgt), 'Партнер': partner, 'Артикул': sku,
                         'prediction': float(preds[i])})
df_out = pd.DataFrame(out_rows)

# Attach observed target_qty for sanity-check
abt_lookup = abt[['Период','Партнер','Артикул','target_qty_imputed']].copy()
abt_lookup['Период'] = abt_lookup['Период'].astype(str)
df_out = df_out.merge(abt_lookup, on=['Период','Партнер','Артикул'], how='left')
df_out['target_qty'] = df_out['target_qty_imputed'].fillna(0)
df_out = df_out[['Период','Партнер','Артикул','target_qty','prediction']]

val_mask = df_out['Период'].isin([str(p) for p in val_periods])
test_mask = df_out['Период'].isin([str(p) for p in test_periods])

df_out[val_mask].to_csv('/kaggle/working/preds_v10_chronos_val.csv', index=False)
df_out[test_mask].to_csv('/kaggle/working/preds_v10_chronos_test.csv', index=False)
print('val rows:', val_mask.sum(), 'test rows:', test_mask.sum())
print(df_out.head())

In [ ]:
import json
def quick_metrics(df):
    a = df['target_qty'].to_numpy()
    p = df['prediction'].to_numpy()
    wape = float(np.abs(a - p).sum() / max(a.sum(), 1e-6))
    bias = float((p.sum() - a.sum()) / max(a.sum(), 1e-6) * 100)
    return {'WAPE': round(wape, 4), 'bias_pct': round(bias, 2)}
print('VAL :', quick_metrics(df_out[val_mask]))
print('TEST:', quick_metrics(df_out[test_mask]))